[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/estimation/multiview/multiview_reconstruction.ipynb)

# Multi-View Reconstruction and Camera Alignment

Scene reconstruction and camera alignment recover the positions of observed landmarks and the poses of the cameras that view them from image pixels alone. Several cameras photograph the same landmarks from different vantage points, with each image recording only the direction from its pinhole toward each landmark. Where sight lines from multiple cameras cross fixes each landmark in space, and when a camera pose is misplaced its sight lines miss those of the other views, driving the misplaced camera back into alignment.

A pinhole camera is modelled as the linear map `(pinhole & Point) ^ sensor`, joining each scene point to the pinhole and meeting the sensor plane. Feeding the camera into a pixel's quadratic cost `ball` on the sensor and pulling polar planes back through `camera.adjugate()` lifts pixel uncertainty into a sight cone `camera.adjugate()(ball(camera))`, even though projection drops a dimension. Carrying each cone into the world as `pose >> cone(pose << Point)` and summing across views forms a fused splat whose spatial polar vanishes at the landmark, so `(splats + w * (w & Point)).solve(w)` recovers its position in closed form while Gauss-Newton alignment evaluates cone curvature and slope directly on `-Twist.commutator(local_centres)`.

In the matrix notation of projective vision, a calibrated camera matrix $P = K\,[R \mid \mathbf t]$ projects homogeneous scene points $\tilde{\mathbf X}$ through perspective division $\pi(\mathbf Y) = (Y_1/Y_3,\, Y_2/Y_3)^{\mathsf T}$, and back-projecting a sensor precision matrix $W$ forms the homogeneous sight-cone matrix $P^{\mathsf T} W P$. Updating camera poses by Gauss-Newton chains the perspective-division Jacobian $\partial\pi/\partial\mathbf Y$ with the skew-matrix rigid-motion block $[-[R\mathbf X + \mathbf t]_\times \;\; I]$ at each landmark to assemble reprojection residuals $\mathbf r$ and normal equations $(J^{\mathsf T}\Sigma^{-1}J)\,\delta\boldsymbol\xi = -J^{\mathsf T}\Sigma^{-1}\mathbf r$.

In [ ]:
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ git+https://github.com/EelcoHoogendoorn/numga"

In [ ]:
%matplotlib inline
from collections.abc import Iterator
from itertools import islice

import numpy as np

from numga import NumpyContext, stack
from numga.algebras import PGA2D
from examples.util.animation import save_figure, show_states
from examples.estimation.multiview import render

np.set_printoptions(precision=4, suppress=True)

# The multiview algebra is written for any projective geometric algebra; this notebook works in 2D.
ga = PGA2D
mv = NumpyContext(ga).multivector

Scalar = ga.gatype.scalar()                         # Scalar+
Point = ga.gatype.antivector()                      # Antivector-
# Ideal points represent weightless displacements between points.
Direction = Point.derive.degenerate                 # Antivector-
Plane = ga.gatype.vector()                          # Vector+
Motor = ga.gatype.rotor()                           # Rotor+
Twist = ga.gatype.bivector()                        # Bivector+
Camera = ga.gatype((Point, Point))                  # Point- <- Point-
Quadric = ga.gatype((Plane, Point))                 # Plane+ <- Point-
Information = ga.gatype((Scalar, Twist, Twist))     # Scalar+ <- (Twist+, Twist+)

## Camera rig and pinhole projection

Six landmarks stand in front of three identical pinhole cameras, with the left and right cameras turned inward and the middle camera looking straight ahead. Each camera is carried from the origin to its world position and heading by a rigid motor.

A pinhole camera projects a scene point to the intersection of its sight line through the pinhole with the sensor plane, `(pinhole & point) ^ sensor`. Writing the point type `Point` as the identity map gives the camera itself as `(pinhole & Point) ^ sensor`, a linear map from scene points to image points.

In [ ]:
landmark_coordinates = [[0.15, 0.85], [-0.43, 1.15], [0.50, 1.50], [0.03, 1.85], [0.65, 2.20], [-0.60, 2.55]]
camera_offsets = np.array([-0.75, 0.75, 0.0])
camera_turns = np.radians([18.0, -18.0, 0.0])

In [ ]:
landmarks = mv.point(landmark_coordinates)                                      # [6] Point-
rig_poses = (mv.xw * (camera_offsets / 2)).exp() * (mv.xy * (camera_turns / 2)).exp()  # [3] Motor+

# Place the canonical pinhole at the origin and the sensor plane one unit ahead along y.
pinhole = mv.point([0, 0])                                                      # [] Point-
sensor = mv.y - mv.w                                                            # [] Plane+
camera = (pinhole & Point) ^ sensor                                             # [] Point- <- Point-
save_figure(render.rig_figure(rig_poses), "multiview_rig");

The blue, pink, and purple wedges show the fields of view of the left, right, and middle cameras, each bounded by its outer sight lines with the sensor bar across the wedge and a dashed optical axis extending forward.

## Pixels and sight cones

Each camera records each landmark at an uncertain pixel on its sensor. On a planar sensor, that uncertainty is modelled as the squared transverse distance from the plane through the pixel perpendicular to the sensor, represented as a polarity map taking any point to its polar plane so that `ball(point) & point` evaluates the quadratic cost.

Composing the sensor quadric with `camera` and pulling its polar planes back through `camera.adjugate()` lifts each pixel's ball into a sight cone `camera.adjugate()(ball(camera))` in the camera frame. Because `camera.adjugate()` transfers incidence across `camera` via `camera.adjugate()(plane) & point == plane & camera(point)`, the pullback exists even though projection drops the depth dimension, and `rig_poses >> cones(rig_poses << Point)` carries every cone into the world.

In the matrix notation of projective vision, `camera.adjugate()` corresponds to the camera-matrix transpose $P^{\mathsf T}$, which maps image lines or planes back to scene planes and lifts a rank-one sensor precision $\mathbf n\,\mathbf n^{\mathsf T}$ to $P^{\mathsf T}\mathbf n\,\mathbf n^{\mathsf T} P$.

In [ ]:
def sensor_balls(pixels: Point) -> Quadric:
    """Planar sensor quadrics (`[...] Plane+ <- Point-`) measure squared transverse distance on the sensor plane from each pixel in `pixels` (`[...] Point-`)."""
    pixel_planes = mv.x - mv.w * (mv.x & pixels)                                # [...] Plane+
    return pixel_planes * (pixel_planes & Point)                                # [...] Plane+ <- Point-


local_landmarks = rig_poses << landmarks[:, None]                               # [6, 3] Point-
images = camera(local_landmarks)                                                # [6, 3] Point-
pixels = images / (mv.w & images)                                               # [6, 3] Point-
balls = sensor_balls(pixels)                                                    # [6, 3] Plane+ <- Point-

# Pull polar planes back from the sensor through the camera adjugate and carry each sight cone into the world.
back_projection = camera.adjugate()                                             # [] Plane+ <- Plane+
cones = back_projection(balls(camera))                                          # [6, 3] Plane+ <- Point-
world_cones = rig_poses >> cones(rig_poses << Point)                            # [6, 3] Plane+ <- Point-
save_figure(render.cones_figure(rig_poses, world_cones), "multiview_cones");

In [ ]:
print(back_projection)

## Closed-form landmark triangulation

Summing a landmark's world sight cones across all cameras forms a single fused quadric splat, compact where sight lines cross steeply and elongated where views are nearly parallel. At the landmark's centre the splat's spatial polar vanishes, and adding the dyad `w * (w & Point)` on the plane at infinity makes that centre the unique pole of `w`, giving the closed-form position `(splats + w * (w & Point)).solve(w)`. Knocking the right camera out of place tilts its cones away from the other two views and spreads the fused splats.

In [ ]:
knock_offsets = np.array([0.7, -0.3])
knock_turn = np.radians(-28.0)

In [ ]:
def triangulation(poses: Motor, cones: Quadric) -> tuple[Point, Quadric]:
    """Carrying local sight `cones` (`[..., ...] Plane+ <- Point-`) into the world by camera `poses` (`[...] Motor+`) and summing across views forms fused splats (`[...] Plane+ <- Point-`) whose centres (`[...] Point-`) reconstruct the landmarks."""
    splats = (poses >> cones(poses << Point)).sum(axis=-1)                      # [...] Plane+ <- Point-
    # Each splat's spatial slope vanishes at its centre, and adding the dyad of the plane at infinity makes the centre its pole.
    centres = (splats + mv.w * (mv.w & Point)).solve(mv.w).normalized()         # [...] Point-
    return centres, splats


def landmark_error(points: Point) -> Scalar:
    """Root-mean-square Euclidean distance (`[] Scalar+`) between unit-weight `points` (`[6] Point-`) and `landmarks`."""
    return (points - landmarks).antinorm().squared().mean(axis=0).square_root()  # [] Scalar+


knock_motor = (((mv.xw * knock_offsets[0] - mv.yw * knock_offsets[1]) / 2).exp()
               * (mv.xy * (knock_turn / 2)).exp())                              # [] Motor+
initial_poses = rig_poses * stack([mv.rotor(), knock_motor, mv.rotor()])        # [3] Motor+
initial_landmarks, initial_splats = triangulation(initial_poses, cones)         # [6] Point-, [6] Plane+ <- Point-
initial_world_cones = initial_poses >> cones(initial_poses << Point)            # [6, 3] Plane+ <- Point-
save_figure(
    render.reconstruction_figure(initial_poses, initial_world_cones, initial_splats, initial_landmarks),
    "multiview_guess",
);

Orange regions show the fused splat level sets around the triangulated landmarks, broadened where the displaced pink camera's sight cones pull away from the blue and purple intersections.

In [ ]:
print(f"initial landmark RMS error {(landmark_error(initial_landmarks) * 100).to_array():.2f} cm")

## Alternating camera alignment

Summing `cones(local_centres) & local_centres` across landmarks and views measures the total geometric mismatch of the scene. Under an infinitesimal camera twist, each landmark in that camera's frame moves by `-Twist.commutator(local_centres)`, a linear map from twists to point motions. Pairing `cones(motion) & motion` forms the Gauss-Newton curvature over pairs of twists and `cones(local_centres) & motion` forms the slope directly from the quadric cost without constructing a residual vector or coordinate Jacobian.

Solving curvature against slope updates the moving camera while the two anchored cameras fix the global rigid gauge and scale of the scene. Because landmarks shift between steps as the camera moves, scaling the twist and capping total pinhole-plus-angular travel walks the displaced camera smoothly from a large initial offset into alignment, while `joint_alignment` and `ray_splat_alignment` in `core` eliminate landmark motion analytically.

In the matrix notation of nonlinear least squares, the single-camera curvature and slope correspond to $J^{\mathsf T} J$ and $J^{\mathsf T}\mathbf r$ built from the residual vector $\mathbf r$ and coordinate Jacobian $J = \partial\mathbf r/\partial\boldsymbol\xi$.

In [ ]:
free_cameras = np.array([0.0, 1.0, 0.0])
stretch, step_cap, landmark_depth = 4.0, 0.08, 1.5
n_steps = 30
frame_duration_ms = 120

In [ ]:
def alignment_steps(
    poses: Motor, cones: Quadric, pinhole: Point, free_cameras: np.ndarray,
    stretch: float, step_cap: float, landmark_depth: float,
) -> Iterator[tuple[Motor, Quadric, Quadric, Point, Information]]:
    """Yields camera poses, world sight cones, fused splats, triangulated landmarks, and single-camera cost curvature across step-capped alternating Gauss-Newton iterations."""
    while True:
        centres, splats = triangulation(poses, cones)
        world_cones = poses >> cones(poses << Point)                            # [..., ...] Plane+ <- Point-
        # Each landmark in each camera's frame moves by minus its commutator with a camera twist.
        local_centres = poses << centres[:, None]                               # [..., ...] Point-
        motion = -Twist.commutator(local_centres)                               # [..., ...] Point- <- Twist+
        curvature = (cones(motion) & motion).sum(axis=0)                        # [...] Scalar+ <- (Twist+, Twist+)
        slope = (cones(local_centres) & motion).sum(axis=0)                     # [...] Scalar+ <- Twist+
        yield poses, world_cones, splats, centres, curvature
        step = curvature.solve(-slope) * free_cameras * stretch                 # [...] Twist+
        # Combine pinhole travel with camera rotation weighted at landmark depth so large initial steps stay bounded.
        step_motors = (step / 2).exp()                                          # [...] Motor+
        step_size = ((step_motors >> pinhole) - pinhole).antinorm() + landmark_depth * step.norm()  # [...] Scalar+
        poses = poses * (step * (step_cap / (step_size + step_cap)) / 2).exp()  # [...] Motor+


def scene_cost(world_cones: Quadric, centres: Point) -> Scalar:
    """Total sight-cone quadratic cost (`[] Scalar+`) evaluated at `centres` (`[6] Point-`) across all views."""
    return (world_cones(centres[:, None]) & centres[:, None]).sum()             # [] Scalar+


estimates = show_states(
    islice(alignment_steps(initial_poses, cones, pinhole, free_cameras, stretch, step_cap, landmark_depth), n_steps + 1),
    render.convergence_animation,
    name="multiview_reconstruction_convergence",
    duration_ms=frame_duration_ms,
    n_frames=n_steps + 1,
    stepped=True,
)

In [ ]:
for index, (_, step_cones, _, step_centres, _) in islice(enumerate(estimates), None, None, 5):
    print(f"step {index:2d}  RMS error {(landmark_error(step_centres) * 100).to_array():.3f} cm  cost {scene_cost(step_cones, step_centres).to_array():.3e}")

## Camera pose uncertainty

Dividing a camera's cost curvature by the pixel variance gives the pose information form over camera twists, measuring how sharply the log-likelihood drops as the camera translates or rotates with the other cameras held fixed. Pushing the inverse information forward through the pinhole's twist motion `poses >> Twist.commutator(pinhole)` forms a dual ellipsoid `Point- <- Plane+` around each pinhole whose inverse quadric bounds the camera's position uncertainty, while evaluating the inverse information on the angular rate `x & Twist.commutator(optical_axis)` gives the heading spread. Drawing both at three standard deviations makes centimetre-and-degree uncertainties visible at the scale of the rig.

In matrix estimation notation on $\mathrm{SE}(2)$, single-camera pose precision is the $3 \times 3$ Fisher information matrix $\mathcal I = \sigma^{-2} J^{\mathsf T} J$, pushing $\mathcal I^{-1}$ through the pinhole Jacobian extracts the $2 \times 2$ position covariance block $\Sigma_{\mathbf t}$ defining $(\mathbf x - \hat{\mathbf t})^{\mathsf T}\Sigma_{\mathbf t}^{-1}(\mathbf x - \hat{\mathbf t}) \le k^2$, and the rotational diagonal entry of $\mathcal I^{-1}$ gives the heading variance.

In [ ]:
pixel_noise = 0.035
n_sigmas = 3.0
optical_axis = mv.direction([0, 1])                                             # [] Direction-

In [ ]:
def pose_uncertainties(
    poses: Motor, curvature: Information, pinhole: Point, transverse_plane: Plane,
    optical_axis: Direction, pixel_noise: float, n_sigmas: float,
) -> tuple[Quadric, Scalar]:
    """Position uncertainty quadrics (`[...] Plane+ <- Point-`) and angular heading spreads (`[...] Scalar+`) at `n_sigmas` standard deviations come from pushing the inverse single-camera `curvature` through pinhole motion and heading rate."""
    information = curvature / pixel_noise**2                                    # [...] Scalar+ <- (Twist+, Twist+)
    pinholes = poses >> pinhole                                                 # [...] Point-
    pinhole_motion = poses >> Twist.commutator(pinhole)                         # [...] Point- <- Twist+
    heading_rate = transverse_plane & Twist.commutator(optical_axis)            # [] Scalar+ <- Twist+
    # Push the pose covariance forward through pinhole motion and centre the dual quadric on each pinhole.
    dual_ellipsoids = (pinhole_motion(information.solve(Plane & pinhole_motion)) * n_sigmas**2
                       - pinholes * (Plane & pinholes))                         # [...] Point- <- Plane+
    ellipsoids = dual_ellipsoids.inverse()                                      # [...] Plane+ <- Point-
    spreads = heading_rate(information.solve(heading_rate)).square_root() * n_sigmas  # [...] Scalar+
    return ellipsoids, spreads


final_poses, final_world_cones, final_splats, final_landmarks, final_curvature = estimates[-1]
ellipsoids, spreads = pose_uncertainties(
    final_poses, final_curvature, pinhole, mv.x, optical_axis, pixel_noise, n_sigmas,
)
save_figure(
    render.covariance_figure(final_poses, final_world_cones, final_splats, final_landmarks, ellipsoids, spreads),
    "multiview_covariance",
);

Shaded ellipses with dashed rims show each camera's three-sigma position uncertainty around its pinhole, and the dotted fans show the corresponding three-sigma heading spread.

In [ ]:
# checks
# Camera adjugate preserves plane-point incidence, and every true landmark lies on all world sight lines.
assert (((back_projection(sensor) & local_landmarks) - (sensor & images)).abs() < 1e-12).all()
assert scene_cost(world_cones, landmarks).abs() < 1e-12

# Displacing the right camera spreads the landmarks, while each splat's spatial slope vanishes at its centre.
assert landmark_error(initial_landmarks) > 0.1
assert (initial_splats(initial_landmarks).norm() < 1e-12).all()

# Alternating alignment recovers the true landmarks and reduces the sight-cone cost by orders of magnitude.
assert landmark_error(final_landmarks) < 1e-3
assert scene_cost(final_world_cones, final_landmarks) < 1e-4 * scene_cost(initial_world_cones, initial_landmarks)

# Each camera's position ellipsoid is centred on its pinhole with positive heading spread.
final_pinholes = final_poses >> pinhole                                         # [3] Point-
assert (ellipsoids(final_pinholes).norm() < 1e-10).all()
assert ((ellipsoids(final_pinholes) & final_pinholes) < 0).all()
assert (spreads > 0).all()